# Fraud Simulator — In-Region Latency (Container Runtime)

Runs the OFS fraud simulator **inside Snowflake (Container Runtime)** so feature reads use
in-region networking (no internet round-trip). Measures:
- OFS feature-serving p50/p95/p99 (300 reads)
- Single end-to-end prediction (OFS + Gateway)
- Concurrent load test (TPS + latency breakdown)
- Sustained QPS load (multiprocess, paced workers)

In [1]:
import os
# Load PAT from Snowflake config
import tomllib, pathlib
#_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')

## 1. Setup

In [2]:
import os, sys, time, random
import numpy as np
import snowflake.connector
from snowflake.snowpark import Session

CONNECTION_NAME = 'parker_demo'
DATABASE        = 'GRAPH_FRAUD_DEMO'
FEATURE_SCHEMA  = 'FEATURES'
WAREHOUSE       = 'GRAPH_FRAUD_WH'

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name=CONNECTION_NAME)
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Connected: {session.get_current_account()}')

In [ ]:
# Column orders per FV (OFS REST returns positional arrays, no column names)
FV_COLUMNS = {
    'GRAPH_COMMUNITY_FEATURES': [
        'COMMUNITY_ID', 'COMMUNITY_SIZE', 'IS_LARGE_COMMUNITY', 'DEGREE',
        'SHARED_ADDRESS_COUNT', 'SHARED_IP_COUNT', 'SHARED_DEVICE_COUNT', 'SHARED_BANK_COUNT',
        'AVG_COMMUNITY_REFUND', 'COMMUNITY_REFUND_STDDEV',
        'ADDR_N_FILERS', 'ADDR_COMMUNITY_SIZE', 'ADDR_AVG_REFUND',
        'IP_N_FILERS', 'IP_COMMUNITY_SIZE', 'IP_AVG_REFUND',
    ],
    'ADDRESS_COMMUNITY_FEATURES': [
        'NUM_FILERS_AT_ADDRESS', 'COMMUNITY_SIZE_AT_ADDRESS', 'AVG_REFUND_AT_ADDRESS',
    ],
    'IP_COMMUNITY_FEATURES': [
        'NUM_FILERS_AT_IP', 'COMMUNITY_SIZE_AT_IP', 'AVG_REFUND_AT_IP',
    ],
}

def query_ofs(fv_name, fv_version, entity_key_name, entity_key_value):
    """Read one entity from OFS REST Query API. Returns (features_dict, latency_ms)."""
    t0 = time.time()
    resp = requests.post(
        f'{QUERY_URL}/api/v1/query',
        headers=HEADERS,
        json={
            'name': fv_name, 'version': fv_version,
            'object_type': 'feature_view',
            'request_rows': [{'entity': {entity_key_name: str(entity_key_value)}}],
        },
        timeout=5,
    )
    ms = (time.time() - t0) * 1000
    row = {}
    if resp.status_code == 200:
        data = resp.json()
        results = data.get('results', data.get('rows', []))
        if results:
            feats = results[0].get('features', {})
            if isinstance(feats, dict):
                row = feats
            elif isinstance(feats, list):
                cols = FV_COLUMNS.get(fv_name, [])
                if len(cols) == len(feats):
                    row = dict(zip(cols, feats))
                else:
                    row = {f'f{i}': v for i, v in enumerate(feats)}
    return row, ms


## 2. Sample Applicants (+ Warm Up)

When running in-region (Container Runtime), warm-up is unnecessary — latency is stable from the first call.
We sample applicants with their filing addresses and IPs for the full feature lookup.

In [4]:
# Sample applicants and warm the connections.
rows = session.sql(f'''
    SELECT t.APPLICANT_ID, r.ADDRESS_ID, r.IP_ADDRESS
    FROM {DATABASE}.RAW.APPLICANTS t
    JOIN {DATABASE}.RAW.APPLICATIONS r ON t.APPLICANT_ID = r.APPLICANT_ID
    LIMIT 500
''').collect()
applicant_data = [(str(r[0]), str(r[1]), str(r[2])) for r in rows]
applicant_ids = [t[0] for t in applicant_data]
print(f'{len(applicant_data)} applicant filings sampled')

# Warm-up: 3 OFS + Gateway calls
for tid, aid, ip in applicant_data[:3]:
    _ = score_one_app(tid, aid, ip, 5000.0)
print('Warm-up complete.')

## 3. OFS Feature-Serving Latency (headline)

Warm, in-region p50/p95/p99 for online feature reads across all 4 backing feature views.
Compare to laptop numbers — the gap is pure network round-trip.

In [5]:
# OFS feature-serving latency benchmark (warm, in-region).
# Reads graph_fv directly to benchmark OFS read latency.
def pct(a, p):
    a = sorted(a)
    return a[min(int(len(a) * p / 100), len(a) - 1)]

N = 300
lats = []
for i in range(N):
    tid, aid, ip = random.choice(applicant_data)
    t0 = time.time()
    try:
        fs.read_feature_view(graph_fv, keys=[[str(tid)]], store_type='online')
    except Exception:
        pass
    wall_ms = (time.time() - t0) * 1000
    lats.append(wall_ms)

print(f'OFS Feature Serving — {N} reads (graph_fv single key)')
print(f'  p50 : {pct(lats, 50):>6.0f} ms')
print(f'  p95 : {pct(lats, 95):>6.0f} ms')
print(f'  p99 : {pct(lats, 99):>6.0f} ms')
print(f'  mean: {sum(lats)/len(lats):>6.0f} ms')

## 4. Single End-to-End Prediction

In [6]:
# Single end-to-end prediction: OFS reads + Gateway with graph FV auto-fetch.
tid, aid, ip = random.choice(applicant_data)
claim_amount = float(np.random.uniform(5000, 30000))
print(f'Applicant: {tid}  Address: {aid}  IP: {ip}  Claim: ${claim_amount:,.2f}')

prob, latency_ms = score_one_app(tid, aid, ip, claim_amount)
decision = 'DECLINE' if prob and prob >= 0.7 else 'REVIEW' if prob and prob >= 0.3 else 'APPROVE'

print(f'\nEnd-to-end latency: {latency_ms:.0f}ms')
print(f'Fraud probability: {prob}  ->  {decision}')

## 5. Concurrent Load Test (full pipeline)

In [7]:
# Concurrent load test: OFS reads + Gateway scoring under concurrency.
from concurrent.futures import ThreadPoolExecutor, as_completed

def score_one(applicant_data_row):
    tid, aid, ip = applicant_data_row
    refund = float(np.abs(np.random.lognormal(8, 0.8)))
    try:
        prob, lat = score_one_app(tid, aid, ip, refund)
        if prob is None:
            return None, lat, 0.0, 'No response'
        decision = 'DECLINE' if prob >= 0.7 else 'REVIEW' if prob >= 0.3 else 'APPROVE'
        return decision, lat, prob, None
    except Exception as e:
        return None, 0, 0.0, str(e)

N, CONCURRENCY = 100, 10
tasks = [random.choice(applicant_data) for _ in range(N)]
gw_lats, decisions, errors = [], {'APPROVE': 0, 'REVIEW': 0, 'DECLINE': 0}, 0

t0 = time.time()
with ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    futs = [pool.submit(score_one, t) for t in tasks]
    for fut in as_completed(futs):
        dec, lat, prob, err = fut.result()
        if err:
            errors += 1
        else:
            gw_lats.append(lat)
            if dec in decisions: decisions[dec] += 1
elapsed = time.time() - t0
completed = len(gw_lats)
tps = completed / max(elapsed, 0.001)

print(f'Load Test: {completed}/{N} successful | {tps:.1f} TPS | {errors} errors')
print(f'\n  {"Latency (ms)":<20} {"Avg":>8} {"p50":>8} {"p95":>8} {"p99":>8}')
print(f'  {"─" * 52}')
if gw_lats:
    avg = sum(gw_lats) / len(gw_lats)
    print(f'  {"Gateway (e2e)":<20} {avg:>8.0f} {pct(gw_lats,50):>8.0f} {pct(gw_lats,95):>8.0f} {pct(gw_lats,99):>8.0f}')
print(f'\n  Decisions: ' + '  '.join(f'{k}={v}' for k, v in decisions.items()))

## 6. Sustained QPS Load — multiprocess (direct REST, benchmark-kit method)

**Key finding:** a single Python process cannot generate 1000+ QPS from the notebook kernel.
We use the `qps_loadgen.proc_worker` function with `ProcessPoolExecutor` to spread the load
across CPU cores, each driving paced threaded workers.

In [8]:
# QPS load generator — threaded, paced, with proper future tracking
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_async_qps(target_qps, duration_s=15, warmup_s=5):
    interval = 1.0 / target_qps
    total_requests = int(target_qps * duration_s)
    warmup_requests = int(target_qps * warmup_s)
    all_requests = warmup_requests + total_requests
    n_workers = min(32, max(4, target_qps // 10))

    print(f'Target: {target_qps} QPS | {n_workers} threads | '
          f'{warmup_requests} warmup + {total_requests} measured')

    def single_read(idx):
        tid = random.choice(applicant_data)[0]
        t0 = time.time()
        try:
            fs.read_feature_view(graph_fv, keys=[[str(tid)]], store_type='online')
            return idx, (time.time() - t0) * 1000, None
        except Exception as e:
            return idx, (time.time() - t0) * 1000, str(e)

    futures = []
    start = time.time()
    pool = ThreadPoolExecutor(max_workers=n_workers)
    for idx in range(all_requests):
        futures.append(pool.submit(single_read, idx))
        time.sleep(interval)

    # Wait for all futures to complete
    latencies = []
    errors = 0
    for f in as_completed(futures):
        idx, lat, err = f.result()
        if err:
            errors += 1
        elif idx >= warmup_requests:
            latencies.append(lat)
    pool.shutdown(wait=False)
    wall = time.time() - start

    if latencies:
        actual_qps = len(latencies) / duration_s
        print(f'\nResults ({len(latencies)} measured, {errors} errors, {wall:.1f}s wall):')
        print(f'  Actual QPS: {actual_qps:.0f}')
        print(f'  p50 : {pct(latencies, 50):>8.1f} ms')
        print(f'  p95 : {pct(latencies, 95):>8.1f} ms')
        print(f'  p99 : {pct(latencies, 99):>8.1f} ms')
        print(f'  mean: {sum(latencies)/len(latencies):>8.1f} ms')
    else:
        print(f'No successful measured requests ({errors} errors)')
    return latencies

print('QPS generator defined: run_async_qps(target_qps, duration_s, warmup_s)')

In [9]:
# 500 QPS (multiprocess, paced workers)
_ = run_async_qps(500, duration_s=15, warmup_s=5)

In [10]:
# 1000 QPS (multiprocess, paced workers)
_ = run_async_qps(1000, duration_s=15, warmup_s=5)

In [11]:
# 1500 QPS (multiprocess, paced workers)
_ = run_async_qps(1500, duration_s=15, warmup_s=5)